### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="fitness_club",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/ddosad/datacamps-data-science-associate-certification",
    download_description="""
kaggle datasets download -d ddosad/datacamps-data-science-associate-certification -p local-data-warehouse/fitness_club/ --unzip
""",
    # References
    academic_reference_bibtex=r"""@misc{ddosad2023fitness,
  author       = {Kaggle User Ddosad},
  title        = {Fitness Club Dataset for ML Classification},
  year         = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/ddosad/datacamps-data-science-associate-certification}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="ddosad2023fitness",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We dropped the booking_id column.
- We renamed the values of the target variable to be more meaningful ("Yes"/"No").
- We treat "-0" values as "0" values for the target, following the description.
- We removed trailing words (like "days") from "days_before".
- We aligned the naming of "day_of_week" to be consistent per day.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="attended",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="attended",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/fitness_class_2212.csv")

df = df.drop(columns=["booking_id"])

target_feature = "attended"
df[target_feature] = abs(df[target_feature])
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"})

# Remove trailing text from column
df["days_before"] = df["days_before"].str.replace(" days", "").astype(int)
df["day_of_week"] = (
    df["day_of_week"]
    .str.replace("Wednesday", "Wed")
    .replace("Monday", "Mon")
    .replace("Fri.", "Fri")
)

cat_features = [
    "day_of_week",
    "time",
    "category",
    "attended",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()